### Setup envs

In [1]:
import os
import logging
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

### Model definition

In [2]:
class UserModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.gender_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_genders'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_genders']) + 1, 4),
        ])

        self.lang_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_langs'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_langs']) + 1, 10),
        ])

        self.country_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_countries'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_countries']) + 1, 10),
        ])

        self.network_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_networks'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_networks']) + 1, 4),
        ])

        age_boundaries = np.array(conf['age_boundaries'])
        self.viewer_age_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.Discretization(age_boundaries.tolist()),
            tf.keras.layers.Embedding(len(age_boundaries), 2)
        ])

        self.centroids = tf.constant(conf['centroids'])
        self.viewer_lat_long_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.TextVectorization(
                standardize = None, split = self.classify,
                vocabulary = [str(i) for i in range(len(self.centroids))],
                max_tokens=len(self.centroids) + 2
                ),
            tf.keras.layers.Embedding(len(self.centroids) + 2, 2),
        ])

    @tf.function()
    def call(self, inputs):
        return tf.concat([
            self.gender_embedding(inputs["viewer_gender"]),
            self.lang_embedding(inputs["viewer_lang"]),
            self.country_embedding(inputs["viewer_country"]),
            self.network_embedding(inputs["viewer_network"]),
            self.viewer_age_embedding(inputs["viewer_age"]),
            self.viewer_lat_long_embedding(inputs["viewer_lat_long"]),
        ], axis = 1)

    @tf.keras.utils.register_keras_serializable()
    def classify(self, pair):
        """
        given a datapoint, compute the cluster closest to the datapoint. Return the cluster ID of that cluster.
        :param pair:
        :return: cluster ID
        """
        str_data = tf.strings.split(pair, sep = ",").values
        str_data = tf.map_fn(lambda x: tf.strings.regex_replace(x, "b'", ""), str_data)
        datapoints = tf.map_fn(lambda x: tf.strings.to_number(x), str_data, dtype = (tf.float32))
        datapoints = tf.reshape(datapoints, [-1, 2])

        expanded_centroids = tf.expand_dims(self.centroids, 1)
        expanded_vectors = tf.expand_dims(datapoints, 0)
        distances = tf.reduce_sum(tf.square(tf.subtract(expanded_vectors, expanded_centroids)), 2)
        clusters = tf.math.argmin(distances)
        return tf.strings.as_string(clusters)

In [3]:
class QueryModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		# We first use the user model for generating embeddings.
		self.embedding_model = UserModel(conf)
		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

In [4]:
class BroadcasterModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.broadcaster_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_broadcasters'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_broadcasters']) + 1, conf['broadcaster_embedding_dimension'])
        ])

    def call(self, broadcaster):
        return tf.concat([
            self.broadcaster_embedding(broadcaster),
        ], axis=1)

In [5]:
class CandidateModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		self.embedding_model = BroadcasterModel(conf)

		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dropout(0.5),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

### Load data

In [6]:
def load_data_file_cold(file, stats):
    print('loading file:' + file)
    training_df = pd.read_csv(
        file,
        skiprows=[0],
        names=["viewer",
               "broadcaster",
               "viewer_age",
               "viewer_gender",
               "viewer_longitude",
               "viewer_latitude",
               "viewer_lang",
               "viewer_country",
               "broadcaster_age",
               "broadcaster_gender",
               "broadcaster_longitude",
               "broadcaster_latitude",
               "broadcaster_lang",
               "broadcaster_country",
               "duration", 
               "viewer_network", 
               "broadcaster_network", 
               "count"], 
        dtype={
            'viewer': np.unicode,
            'broadcaster': np.unicode,
            'viewer_age': np.single,
            'viewer_gender': np.unicode,
            'viewer_longitude': np.single,
            'viewer_latitude': np.single,
            'viewer_lang': np.unicode,
            'viewer_country': np.unicode,
            'broadcaster_age': np.single,
            'broadcaster_longitude': np.single,
            'broadcaster_latitude': np.single,
            'broadcaster_lang': np.unicode,
            'broadcaster_country': np.unicode,
            'duration': np.single,
            'viewer_network': np.unicode,
            'broadcaster_network': np.unicode,
            'count': np.unicode,
        })

    values = {
        'viewer': 'unknown',
        'broadcaster': 'unknown',
        'viewer_age': 30,
        'viewer_gender': 'unknown',
        'viewer_longitude': 0,
        'viewer_latitude': 0,
        'viewer_lang': 'unknown',
        'viewer_country': 'unknown',
        'broadcaster_age': 30,
        'broadcaster_longitude': 0,
        'broadcaster_latitude': 0,
        'broadcaster_lang': 'unknown',
        'broadcaster_country': 'unknown',
        'duration': 0,
        'viewer_network': 'unknown',
        'broadcaster_network': 'unknown',
        "viewer_lat_long": tf.constant(["40.36393,-74.89611"]),
        'count': '1'
    }

    training_df = training_df.sample(frac=0.00001)
    training_df.fillna(value=values, inplace=True)
    training_df['viewer_lat_long'] = training_df[['viewer_latitude', 'viewer_longitude']].apply(lambda x: '{},{}'.format(x[0],x[1]), axis=1)
    training_df['duration'] = np.log(1 + training_df['duration'])
    print(training_df.head(10))
    print(training_df.iloc[-10:])
    # stats.send_stats('data-size', len(training_df.index))
    return training_df


def load_training_data_cold(file, stats):
    ratings_df = load_data_file_cold(file, stats)
    print('creating data set')
    training_ds = (
        tf.data.Dataset.from_tensor_slices(
            ({
                "viewer": tf.cast(
                    ratings_df['viewer'].values,
                    tf.string),
                "viewer_gender": tf.cast(
                    ratings_df['viewer_gender'].values,
                    tf.string),
                "viewer_lang": tf.cast(
                    ratings_df['viewer_lang'].values,
                    tf.string),
                "viewer_country": tf.cast(
                    ratings_df['viewer_country'].values,
                    tf.string),
                "viewer_age": tf.cast(
                    ratings_df['viewer_age'].values,
                    tf.int32),
                "viewer_longitude": tf.cast(
                    ratings_df['viewer_longitude'].values,
                    tf.float16),
                "viewer_latitude": tf.cast(
                    ratings_df['viewer_latitude'].values,
                    tf.float16),
                "broadcaster": tf.cast(
                    ratings_df['broadcaster'].values,
                    tf.string),
                "viewer_network": tf.cast(
                    ratings_df['viewer_network'].values,
                    tf.string),
                "broadcaster_network": tf.cast(
                    ratings_df['broadcaster_network'].values,
                    tf.string),
                "duration": tf.cast(
                    ratings_df['duration'].values,
                    tf.float16),
                "viewer_lat_long": tf.cast(
                    ratings_df['viewer_lat_long'].values,
                    tf.string),
            })))

    return training_ds
            

def prepare_training_data_cold(train_ds):
    print('prepare_training_data')
    training_ds = train_ds.cache().map(lambda x: {
        "broadcaster": x["broadcaster"],
        "viewer": x["viewer"],
        "viewer_gender": x["viewer_gender"],
        "viewer_lang": x["viewer_lang"],
        "viewer_country": x["viewer_country"],
        "viewer_age": x["viewer_age"],
        "viewer_longitude": x["viewer_longitude"],
        "viewer_latitude": x["viewer_latitude"],
        "viewer_network": x["viewer_network"],
        "broadcaster_network": x["broadcaster_network"],
        "duration": x["duration"],
        "viewer_lat_long": x["viewer_lat_long"],
    }, num_parallel_calls=tf.data.AUTOTUNE,
       deterministic=False)

    print('done prepare_training_data')
    return training_ds

In [7]:
def get_broadcaster_data_set(train_ds):
    broadcasters = train_ds.cache().map(lambda x: x["broadcaster"], 
                                        num_parallel_calls=tf.data.AUTOTUNE, 
                                        deterministic=False)
    broadcasters_ds = tf.data.Dataset.from_tensor_slices(
        np.unique(list(broadcasters.as_numpy_iterator())))
    return broadcasters_ds

In [8]:
training_dataset = load_training_data_cold("csv/2022-01-03.csv", "")

loading file:csv/2022-01-03.csv


2022-01-25 16:03:00.877535: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


                                                  viewer  \
5577581  aa 7c c0 a7 90 76 21 fd e5 fd 3b 24 50 96 24 ac   
1029016  34 1e de 18 76 02 eb 4d 80 a6 df 76 58 0c 4d cf   
4662784  d2 22 dc 5f df 32 02 01 8c d1 85 7e cd 78 8e 8c   
5562157  82 c5 18 15 8d d0 da ce 91 48 7f e5 4d a9 97 27   
5729524  34 f1 0b 0d ce c7 ba 07 90 7a 35 99 70 b0 67 30   
8503974  d1 29 57 7b 91 65 bd 2e 7d 72 39 98 a2 63 28 a2   
6671309  68 2d 06 b4 ad 72 71 59 f9 7d 97 b3 74 02 0a 09   
1799432  6d 76 b1 6c b4 3c cd 16 92 1c 38 c9 c5 eb d6 c0   
334817   0f f4 75 ac 96 42 b3 6d 6f 77 f5 0c 6e d0 fe e9   
8317414  12 f5 65 d8 ea 70 23 9e 8a 2f f8 bb 32 6d 09 c4   

                                             broadcaster  viewer_age  \
5577581  09 d3 42 ef 69 95 9b 8b 75 77 3f 47 8e 26 5f 94        28.0   
1029016  5b 64 48 b4 ee 07 e6 39 fb 6a 9a 6a 11 07 85 bc        30.0   
4662784  b9 25 e1 54 cf c6 1a 1d 51 77 5c 1e 4b 3f 85 74        39.0   
5562157  37 77 2b 3a e2 ff 25 30 04 43 ee c6 c8 35 

In [9]:
train = prepare_training_data_cold(training_dataset)

prepare_training_data
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
done prepare_training_data


In [10]:
broadcasters_data_set = get_broadcaster_data_set(training_dataset)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2022-01-25 16:03:02.045188: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)


### Prepare model conf

In [11]:
def get_list(training_data, key):
    return training_data.batch(1_000_000).map(lambda x: x[key], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)


def get_unique_list(data):
    return np.unique(np.concatenate(list(data)))

In [12]:
user_genders = get_list(train, 'viewer_gender')

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [13]:
user_langs = get_list(train, 'viewer_lang')

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [14]:
user_countries = get_list(train, 'viewer_country')

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [15]:
viewer_age = get_list(train, 'viewer_age')

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [16]:
user_networks = get_list(train, 'viewer_network')

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


### derive input dims

In [17]:
unique_user_genders = get_unique_list(user_genders)

In [18]:
len(unique_user_genders)

2

In [19]:
unique_user_langs = get_unique_list(user_langs)

In [20]:
len(unique_user_langs)

8

In [21]:
unique_user_countries = get_unique_list(user_countries)

In [22]:
len(unique_user_countries)

11

In [23]:
unique_user_networks = get_unique_list(user_networks)

In [24]:
len(unique_user_networks)

4

In [25]:
broadcaster_ids = get_list(train, 'broadcaster')

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


In [26]:
unique_broadcasters = get_unique_list(broadcaster_ids)

In [27]:
len(unique_broadcasters)

89

In [28]:
broadcaster_embedding_dimension = 32

In [29]:
cold_start_conf = {
    'unique_genders': unique_user_genders,
    'unique_langs': unique_user_langs,
    'unique_countries': unique_user_countries,
    'unique_networks': unique_user_networks,
    'unique_broadcasters': unique_broadcasters,
    'broadcaster_embedding_dimension': broadcaster_embedding_dimension,
    'age_boundaries': [18, 25, 30, 35, 40, 45, 50, 55, 60, 65, float("inf")],
    'centroids': [[36.68147669256268, -82.8910274009993],
        [23.22243322909555, 78.23027450833709],
        [50.04997682638993, 0.22379313938744885],
        [37.9309447099281, -117.00741350764692],
        [-32.795864819917725, 148.7159172660312],
        [-18.570548393114084, -54.280255665692565],
        [13.921140442819565, 116.38740315555172],
        [29.78951080730802, 40.279515865947936]]
}

In [30]:
cold_start_conf

{'unique_genders': array([b'female', b'male'], dtype=object),
 'unique_langs': array([b'de', b'en', b'es', b'id', b'ja', b'pt', b'th', b'zh'],
       dtype=object),
 'unique_countries': array([b'AU', b'BR', b'CR', b'DE', b'ES', b'ID', b'JP', b'PR', b'TH',
        b'TW', b'US'], dtype=object),
 'unique_networks': array([b'meetme', b'pof', b'skout', b'zoosk'], dtype=object),
 'unique_broadcasters': array([b'00 0e 9e f8 31 b4 ab 8e 71 68 05 95 85 f9 11 0e',
        b'03 7f 98 b3 14 8f 8e d9 75 b5 14 d5 45 4a 17 67',
        b'03 d6 a4 c7 5f e7 8d a6 3b bc 73 54 7b 54 29 ac',
        b'04 41 3c 59 33 f8 bc 8a a1 07 ef 35 06 f1 cb 25',
        b'09 0c 33 f2 99 a1 10 97 6f 53 78 d0 0c 90 db 39',
        b'09 d3 42 ef 69 95 9b 8b 75 77 3f 47 8e 26 5f 94',
        b'10 6f b3 8e f7 2e ec 76 5c bb 32 cf 41 5c a7 b4',
        b'13 41 c8 37 a9 16 32 6e d1 0c c4 6f a3 21 f6 86',
        b'15 da 37 63 69 80 48 1e dc dd bc d7 46 ea b5 d8',
        b'20 4f 4e f7 5b 1f 3f 96 6a f9 a7 df ab f8 ac 96',
 

### query model

In [31]:
query_model = QueryModel(cold_start_conf)

### broadcaster model

In [32]:
candidate_model = CandidateModel(cold_start_conf)

### Candidate / Ranking model

In [33]:
class RankingModel(tf.keras.Model):

	def __init__(self):
		super().__init__()
		embedding_dimension = 32

		# Compute predictions.
		self.ratings = tf.keras.Sequential(
			[
				# Learn multiple dense layers.
				tf.keras.layers.Dense(256, activation = "relu"),
				tf.keras.layers.Dense(64, activation = "relu"),
				# Make rating predictions in the final layer.
				tf.keras.layers.Dense(1)
			]
		)

	def call(self, inputs):
		query_embeddings, positive_broadcaster_embeddings = inputs
		return self.ratings(tf.concat([query_embeddings, positive_broadcaster_embeddings], axis = 1))

In [34]:
ranking_model = RankingModel()

### Loss and metrics

In [35]:
task = tfrs.tasks.Ranking(
  loss = tf.keras.losses.MeanSquaredError(),
  metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

In [36]:
from typing import Dict, Text

In [37]:
class TwoTowers(tfrs.models.Model) :

    def __init__(self, candidate_model, query_model, ranking_model, task):
        super().__init__()
        self.query_model: tf.keras.Model = query_model
        self.candidate_model: tf.keras.Model = candidate_model
        self.ranking_model: tf.keras.Model = ranking_model
        self.task = task

    def call(self, features: Dict[str, tf.Tensor]) -> tf.Tensor :
        query_embeddings = self.query_model({
            "viewer_gender": features["viewer_gender"],
            "viewer_lang": features["viewer_lang"],
            "viewer_country": features["viewer_country"],
            "viewer_age": features["viewer_age"],
            "viewer_network": features["viewer_network"],
            "viewer_latitude": features["viewer_latitude"],
            "viewer_longitude": features["viewer_longitude"],
            "viewer_lat_long": features["viewer_lat_long"],
        })
        positive_broadcaster_embeddings = self.candidate_model(
            features["broadcaster"])
        return self.ranking_model(
            (query_embeddings, positive_broadcaster_embeddings))

    def compute_loss(self, features: Dict[Text, tf.Tensor], training = False) -> tf.Tensor :
        labels = features["duration"]
        
        query_embeddings = self.query_model({
            "viewer_gender": features["viewer_gender"],
            "viewer_lang": features["viewer_lang"],
            "viewer_country": features["viewer_country"],
            "viewer_age": features["viewer_age"],
            "viewer_network": features["viewer_network"],
            "viewer_latitude": features["viewer_latitude"],
            "viewer_longitude": features["viewer_longitude"],
            "viewer_lat_long": features["viewer_lat_long"],
        })
        positive_broadcaster_embeddings = self.candidate_model(
            features["broadcaster"])
        
        rating_predictions = self.ranking_model(
            (query_embeddings, positive_broadcaster_embeddings))

        # The task computes the loss and the metrics.
        return self.task(labels = labels, predictions = rating_predictions)

In [38]:
model = TwoTowers(candidate_model, query_model, ranking_model, task)

In [39]:
learning_rate = 0.00001
batch_size = 16384
# batch_size = 250
epochs = 20
patience = 2
top_k = 1999

In [40]:
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate))

In [41]:
tf.random.set_seed(42)
shuffled = train.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

train_p80 = shuffled.take(80_000)
test_p20 = shuffled.skip(80_000).take(20_000)

cached_train = train_p80.shuffle(100_000).batch(batch_size)
cached_test = test_p20.batch(2048).cache()

In [42]:
hist = model.fit(cached_train, epochs=10)

Epoch 1/10
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing t

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set

In [43]:
# model.fit(train_ds, epochs=epochs)
callback = tf.keras.callbacks.EarlyStopping(monitor="total_loss",
                                            patience=patience,
                                            verbose = 1,
                                            restore_best_weights = True
                                            )
hist = model.fit(cached_train,
          epochs=epochs,
          validation_data=cached_test,
          validation_freq=1,
          callbacks=[callback]
          )

Epoch 1/20
1/1 [==============================] - 0s 26ms/step - root_mean_squared_error: 3.3932 - loss: 11.5136 - regularization_loss: 0.0064 - total_loss: 11.5200
Epoch 2/20
1/1 [==============================] - 0s 24ms/step - root_mean_squared_error: 3.3931 - loss: 11.5133 - regularization_loss: 0.0064 - total_loss: 11.5198
Epoch 3/20
1/1 [==============================] - 0s 24ms/step - root_mean_squared_error: 3.3931 - loss: 11.5131 - regularization_loss: 0.0064 - total_loss: 11.5195
Epoch 4/20
1/1 [==============================] - 0s 25ms/step - root_mean_squared_error: 3.3931 - loss: 11.5129 - regularization_loss: 0.0064 - total_loss: 11.5193
Epoch 5/20
1/1 [==============================] - 0s 24ms/step - root_mean_squared_error: 3.3930 - loss: 11.5126 - regularization_loss: 0.0064 - total_loss: 11.5190
Epoch 6/20
1/1 [==============================] - 0s 25ms/step - root_mean_squared_error: 3.3930 - loss: 11.5124 - regularization_loss: 0.0064 - total_loss: 11.5188
Epoch 7/20

In [44]:
hist.history

{'root_mean_squared_error': [3.3931705951690674,
  3.3931326866149902,
  3.3930959701538086,
  3.3930606842041016,
  3.3930258750915527,
  3.3929922580718994,
  3.3929595947265625,
  3.392927408218384,
  3.3928964138031006,
  3.3928658962249756,
  3.392835855484009,
  3.3928062915802,
  3.392777442932129,
  3.392749309539795,
  3.39272141456604,
  3.3926944732666016,
  3.392667531967163,
  3.392641067504883,
  3.3926150798797607,
  3.392589569091797],
 'loss': [11.513607025146484,
  11.513349533081055,
  11.513100624084473,
  11.512861251831055,
  11.512624740600586,
  11.512396812438965,
  11.512175559997559,
  11.511957168579102,
  11.51174545288086,
  11.5115385055542,
  11.511335372924805,
  11.51113510131836,
  11.51093864440918,
  11.510747909545898,
  11.51055908203125,
  11.5103759765625,
  11.51019287109375,
  11.510013580322266,
  11.50983715057373,
  11.509663581848145],
 'regularization_loss': [0.006406704429537058,
  0.006406704429537058,
  0.006406704429537058,
  0.006406

### Testing the ranking model

In [46]:
import pprint

In [51]:
for x in cached_train.take(1).as_numpy_iterator():
    pprint.pprint(x['broadcaster'][:3])

array([b'bf 75 7a a5 78 aa 15 f3 db 18 a7 54 c0 d9 37 96',
       b'00 0e 9e f8 31 b4 ab 8e 71 68 05 95 85 f9 11 0e',
       b'77 e3 10 09 dc 21 1d 91 6d 82 fb a9 fb 1a c7 bd'], dtype=object)


In [61]:
test_ratings = {}
test_broadcasters = ['bf 75 7a a5 78 aa 15 f3 db 18 a7 54 c0 d9 37 96',
                    '00 0e 9e f8 31 b4 ab 8e 71 68 05 95 85 f9 11 0e',
                    '77 e3 10 09 dc 21 1d 91 6d 82 fb a9 fb 1a c7 bd']
for bradcaster_id in test_broadcasters:
    test_ratings[bradcaster_id] = model({
    "viewer_gender": np.array([ "female" ]),
            "viewer_lang": np.array(["en"]),
            "viewer_country": np.array(["US"]),
            "viewer_age": np.array([36]),
            "viewer_network": np.array(["pof"]),
            "viewer_latitude": np.array([-36.8]),
            "viewer_longitude": np.array([-84.1]),
            "viewer_lat_long": np.array(["-36.8,-84.1"]),
            "broadcaster": np.array([bradcaster_id])
  })

print("Ratings:")
for bradcaster_id, score in sorted(test_ratings.items(), key=lambda x: x[1], reverse=True):
    print(f"{bradcaster_id}: {score}")

Ratings:
00 0e 9e f8 31 b4 ab 8e 71 68 05 95 85 f9 11 0e: [[0.00076344]]
77 e3 10 09 dc 21 1d 91 6d 82 fb a9 fb 1a c7 bd: [[-0.0029318]]
bf 75 7a a5 78 aa 15 f3 db 18 a7 54 c0 d9 37 96: [[-0.00376532]]


### Exporting for serving

In [76]:
tf.saved_model.save(model, "exported_ranking_model")

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module 'gast' has no attribute 'Constant'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


...

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: 'arguments' object has no attribute 'posonlyargs'
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert

FOR DEVS: If you are overwriting _tracking_metadata in your class, this property has been used to save metadata in the SavedModel. The metadta field will be deprecated soon, so please move the metadata to a different file.



FOR DEVS: If you are overwriting _tracking_metadata in your class, this property has been used to save metadata in the SavedModel. The metadta field will be deprecated soon, so please move the metadata to a different file.


INFO:tensorflow:Assets written to: exported_ranking_model/assets


INFO:tensorflow:Assets written to: exported_ranking_model/assets


In [77]:
loaded_model = tf.saved_model.load("exported_ranking_model")

In [88]:
model({"viewer_gender": np.array(["female"]),
            "viewer_lang": np.array(["en"]),
            "viewer_country": np.array(["US"]),
            "viewer_age": np.array([36]),
            "viewer_network": np.array(["pof"]),
            "viewer_latitude": np.array([-36.8]),
            "viewer_longitude": np.array([-84.1]),
            "viewer_lat_long": np.array(["-36.8,-84.1"]),
            "broadcaster": np.array(["bf 75 7a a5 78 aa 15 f3 db 18 a7 54 c0 d9 37 96"])
})

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[-0.00376532]], dtype=float32)>

In [89]:
loaded_model({"viewer_gender": np.array(["female"]),
            "viewer_lang": np.array(["en"]),
            "viewer_country": np.array(["US"]),
            "viewer_age": np.array([36]),
            "viewer_network": np.array(["pof"]),
            "viewer_latitude": np.array([-36.8]),
            "viewer_longitude": np.array([-84.1]),
            "viewer_lat_long": np.array(["-36.8,-84.1"]),
            "broadcaster": np.array(["bf 75 7a a5 78 aa 15 f3 db 18 a7 54 c0 d9 37 96"])
})

ValueError: Could not find matching function to call loaded from the SavedModel. Got:
  Positional arguments (2 total):
    * {'viewer_gender': <tf.Tensor 'features_3:0' shape=(1,) dtype=string>, 'viewer_lang': <tf.Tensor 'features_4:0' shape=(1,) dtype=string>, 'viewer_country': <tf.Tensor 'features_2:0' shape=(1,) dtype=string>, 'viewer_age': <tf.Tensor 'features_1:0' shape=(1,) dtype=int64>, 'viewer_network': <tf.Tensor 'features_8:0' shape=(1,) dtype=string>, 'viewer_latitude': <tf.Tensor 'features_6:0' shape=(1,) dtype=float64>, 'viewer_longitude': <tf.Tensor 'features_7:0' shape=(1,) dtype=float64>, 'viewer_lat_long': <tf.Tensor 'features_5:0' shape=(1,) dtype=string>, 'broadcaster': <tf.Tensor 'features:0' shape=(1,) dtype=string>}
    * False
  Keyword arguments: {}

Expected these arguments to match one of the following 4 option(s):

Option 1:
  Positional arguments (2 total):
    * {'viewer_network': TensorSpec(shape=(None,), dtype=tf.string, name='viewer_network'), 'broadcas

In [90]:
loaded_models({
'viewer_network': tf.constant(["pof"]), 
    'broadcaster': tf.constant(["bf 75 7a a5 78 aa 15 f3 db 18 a7 54 c0 d9 37 96"]), 
    'viewer_longitude': tf.constant([-84.1]),
    'viewer_gender': tf.constant(["female" ]),
    'viewer_lang': tf.constant(["en"]),
    'viewer_country': tf.constant(["US"]),
    'viewer_age': tf.constant([36]),
    'viewer_lat_long': tf.constant(["-36.8,-84.1"]),
    'viewer_latitude': tf.constant([-36.8])
})

ValueError: Could not find matching function to call loaded from the SavedModel. Got:
  Positional arguments (2 total):
    * {'viewer_network': <tf.Tensor 'features_8:0' shape=(1,) dtype=string>, 'broadcaster': <tf.Tensor 'features:0' shape=(1,) dtype=string>, 'viewer_longitude': <tf.Tensor 'features_7:0' shape=(1,) dtype=float32>, 'viewer_gender': <tf.Tensor 'features_3:0' shape=(1,) dtype=string>, 'viewer_lang': <tf.Tensor 'features_4:0' shape=(1,) dtype=string>, 'viewer_country': <tf.Tensor 'features_2:0' shape=(1,) dtype=string>, 'viewer_age': <tf.Tensor 'features_1:0' shape=(1,) dtype=int32>, 'viewer_lat_long': <tf.Tensor 'features_5:0' shape=(1,) dtype=string>, 'viewer_latitude': <tf.Tensor 'features_6:0' shape=(1,) dtype=float32>}
    * False
  Keyword arguments: {}

Expected these arguments to match one of the following 4 option(s):

Option 1:
  Positional arguments (2 total):
    * {'viewer_latitude': TensorSpec(shape=(None,), dtype=tf.float32, name='viewer_latitude'), 'viewe